# Figure 7: hero scatter (interpreter mean vs stacking-model probability)

Regenerates the four-panel scatter (manuscript Figure 7) comparing the mean of three human interpreters against the predicted probability of each of the four stacking configurations, over the 400 reference points, split by base-learner agreement and disagreement, with a 1:1 line, a linear fit, and an R-squared per panel.

Method note: each model value is read at the centroid pixel at the native 4.77 m PlanetScope resolution (a single-pixel predicted probability), not a buffer average. The interpreter value aggregates the three interpreters within a 10 m buffer, which is the mechanism that matches the same physical point across the three interpreter assets.

## Inputs (Google Earth Engine)
- Six interpreter FeatureCollections: `projects/ee-islamkm/assets/interpreter{1,2,3}_200pts` and `..._200pts_certain`.
- Four stacking-model probability images: `projects/ee-ashrafulcuetbd/assets/stacking_{logreg,rf}_{npt,pt}_prediction`.

Requires a one-time `earthengine authenticate` and read access to those projects.

## Outputs (in `outputs/`)
- `hero_scatter_relabeled.png` (300 dpi) and `.pdf`.
- `reference_points_with_probs.csv`: per-point interpreter means, centroid model probabilities, and stratum. Shipped so the figure can be restyled or verified without re-querying GEE.

## How to run
Python 3.10+ with the packages in `requirements.txt`. Run top to bottom. Sections 8 to 11 (OLS goodness-of-fit, confusion matrices, accuracy metrics) are optional diagnostics that support the R-squared and calibration statements in the text; the figure itself is produced by sections 1 to 7.

## 1. Setup

In [ ]:
# Import required libraries.
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Initialize the Earth Engine API (requires a one-time: earthengine authenticate).
ee.Initialize()

# Results (figure + per-point CSV) are written here, next to this notebook.
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Convert an Earth Engine FeatureCollection to a pandas DataFrame (replaces geemap.ee_to_df).
def ee_fc_to_df(fc):
    info = fc.getInfo()
    rows = [feat.get('properties', {}) for feat in info.get('features', [])]
    return pd.DataFrame(rows)

# R-squared without scikit-learn (replaces sklearn.metrics.r2_score).
def r2_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot else float('nan')

## 2. Expand MultiPoint features

In [ ]:
# Expand MultiPoint or GeometryCollection features into individual point features.
def explode_multi_points(fc):
    def explode_feature(feature):
        geom = feature.geometry()
        geom_type = geom.type()
        is_container = ee.List(['MultiPoint', 'GeometryCollection']).contains(geom_type)

        # For container geometries, create one feature per sub-geometry and copy properties.
        def container():
            geoms = ee.List(geom.geometries())
            return ee.FeatureCollection(
                geoms.map(lambda g: ee.Feature(ee.Geometry(g)).copyProperties(feature))
            )

        # For single geometries, keep the feature unchanged.
        def single():
            return ee.FeatureCollection([feature])

        return ee.FeatureCollection(ee.Algorithms.If(is_container, container(), single()))

    # Flatten the mapped collections into one output FeatureCollection.
    return ee.FeatureCollection(fc.map(explode_feature).flatten())

## 3. Load interpreter point collections

In [ ]:
# Load the "certain" interpreter point collections.
fc1_cert = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter1_200pts_certain')
fc2_cert = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter2_200pts_certain')
fc3_cert = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter3_200pts_certain')

# Expand container geometries into individual point features.
fc1_cert = explode_multi_points(fc1_cert)
fc2_cert = explode_multi_points(fc2_cert)
fc3_cert = explode_multi_points(fc3_cert)

# Load the full interpreter point collections.
fc1_conf = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter1_200pts')
fc2_conf = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter2_200pts')
fc3_conf = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter3_200pts')

# Tag each collection so the source can be tracked later.
fc1_cert = fc1_cert.map(lambda f: f.set('source', 'cert'))
fc1_conf = fc1_conf.map(lambda f: f.set('source', 'conf'))
fc2_cert = fc2_cert.map(lambda f: f.set('source', 'cert'))
fc2_conf = fc2_conf.map(lambda f: f.set('source', 'conf'))
fc3_cert = fc3_cert.map(lambda f: f.set('source', 'cert'))
fc3_conf = fc3_conf.map(lambda f: f.set('source', 'conf'))

# Merge the confident and full collections for each interpreter.
fc1 = fc1_cert.merge(fc1_conf)
fc2 = fc2_cert.merge(fc2_conf)
fc3 = fc3_cert.merge(fc3_conf)

## 4. Load stacked generalization model outputs

In [ ]:
# Load the model prediction rasters.
logreg_npt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_logreg_npt_prediction")
logreg_pt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_logreg_pt_prediction")
rf_npt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_rf_npt_prediction")
rf_pt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_rf_pt_prediction")

## 5. Sampling helpers and point-level statistics

In [ ]:
# Native pixel size of the prediction rasters (about 4.77 m). Computed once.
NATIVE_SCALE = logreg_npt.projection().nominalScale().getInfo()
print(f'Prediction raster native scale: {NATIVE_SCALE:.3f} m')

# Sample the first image band at the point's native pixel. This returns the
# centroid pixel value, not a mean over the 10 m buffer.
def sample_image_at_point(img, geom, scale=NATIVE_SCALE):
    band = ee.String(img.bandNames().get(0))
    d = img.reduceRegion(reducer=ee.Reducer.first(), geometry=geom, scale=scale)
    return ee.Number(d.get(band))


# Compute interpreter and model statistics for each reference point.
def compute_point_stats_v2(feat):
    geom = feat.geometry()
    buf = geom.buffer(10)

    # Interpreter score aggregates the three interpreters within the 10 m buffer.
    # The buffer matches the same physical point across the three interpreter assets.
    i1 = ee.Number(fc1.filterBounds(buf).aggregate_mean('val_int'))
    i2 = ee.Number(fc2.filterBounds(buf).aggregate_mean('val_int'))
    i3 = ee.Number(fc3.filterBounds(buf).aggregate_mean('val_int'))

    interp_mean = i1.add(i2).add(i3).divide(3)
    interp_mean_sq = i1.pow(2).add(i2.pow(2)).add(i3.pow(2)).divide(3)
    interp_std = interp_mean_sq.subtract(interp_mean.pow(2)).max(0).sqrt()

    # Model predicted probability at the centroid pixel (native about 4.77 m),
    # rather than a mean over the 10 m buffer.
    logreg_npt_p = sample_image_at_point(logreg_npt, geom)
    logreg_pt_p = sample_image_at_point(logreg_pt, geom)
    rf_npt_p = sample_image_at_point(rf_npt, geom)
    rf_pt_p = sample_image_at_point(rf_pt, geom)

    return feat.set({
        'source': feat.get('source'),
        'i1_mean': i1,
        'i2_mean': i2,
        'i3_mean': i3,
        'interp_std': interp_std,
        'logreg_npt_prob': logreg_npt_p,
        'logreg_pt_prob': logreg_pt_p,
        'rf_npt_prob': rf_npt_p,
        'rf_pt_prob': rf_pt_p,
    })

## 6. Compute point-level results and convert to a DataFrame

In [ ]:
# Use interpreter 1 as the reference geometry collection.
result_fc = fc1.map(compute_point_stats_v2)

# Convert the Earth Engine FeatureCollection to a pandas DataFrame.
df2 = ee_fc_to_df(result_fc)

# Display the resulting table.
print(f'Rows pulled from GEE: {len(df2)}')
df2.head()

## 7. Plot interpreter means against model predictions

In [ ]:
# Compute the interpreter mean from the three interpreter summaries.
df2['interp_mean'] = df2[['i1_mean', 'i2_mean', 'i3_mean']].mean(axis=1)

# Save the per-point table so the figure can be restyled without re-querying GEE.
df2.to_csv(OUT_DIR / 'reference_points_with_probs.csv', index=False)

# Stacking-configuration columns (model predicted probability at the centroid pixel).
model_cols = ['logreg_npt_prob', 'logreg_pt_prob', 'rf_npt_prob', 'rf_pt_prob']

# Written-out panel titles (collaborator id=33). "Feature pass-through" matches the
# manuscript wording: base-learner features included alongside their predictions.
label_map = {
    'logreg_npt_prob': 'Logistic-regression stacking,\nno feature pass-through',
    'logreg_pt_prob':  'Logistic-regression stacking,\nwith feature pass-through',
    'rf_npt_prob':     'Random-forest stacking,\nno feature pass-through',
    'rf_pt_prob':      'Random-forest stacking,\nwith feature pass-through',
}
model_labels = [label_map[col] for col in model_cols]

# Enlarged fonts (collaborator id=33).
TITLE_FS, AXIS_FS, TICK_FS, LEG_FS, ANNOT_FS = 14, 15, 12, 12, 13
TICKS = np.arange(0, 1.01, 0.1)   # 0.1 increments, labels run 0.0 to 1.0 (no negative)
PAD = 0.04                        # small margin so points at 0 and 1 are fully visible
LO, HI = -PAD, 1 + PAD

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

legend_handles, legend_labels = None, None
for ax, col, label in zip(axes, model_cols, model_labels):
    cert_mask = (df2['source'] == 'cert') & df2['interp_mean'].notna() & df2[col].notna()
    conf_mask = (df2['source'] == 'conf') & df2['interp_mean'].notna() & df2[col].notna()

    ax.scatter(df2.loc[cert_mask, 'interp_mean'], df2.loc[cert_mask, col],
               alpha=0.7, s=40, edgecolor='k', color='steelblue', zorder=3,
               label='Base-learners agree')
    ax.scatter(df2.loc[conf_mask, 'interp_mean'], df2.loc[conf_mask, col],
               alpha=0.7, s=40, edgecolor='k', color='coral', zorder=3,
               label='Less consensus among base-learners')

    x_all = df2.loc[cert_mask | conf_mask, 'interp_mean']
    y_all = df2.loc[cert_mask | conf_mask, col]

    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, zorder=2, label='1:1 line')

    coeffs = np.polyfit(x_all, y_all, deg=1)
    poly = np.poly1d(coeffs)
    x_fit = np.linspace(0, 1, 100)
    ax.plot(x_fit, poly(x_fit), 'b-', linewidth=1.5, zorder=2, label='Linear fit')

    r2 = r2_np(y_all, poly(x_all))

    # Light grey grid, drawn behind the data.
    ax.set_axisbelow(True)
    ax.grid(True, color='lightgray', linewidth=0.6)

    # Square panel; limits padded slightly beyond 0 and 1 so edge points stay
    # fully visible, but tick labels still run 0.0 to 1.0 with no negative values.
    ax.set_xlim(LO, HI)
    ax.set_ylim(LO, HI)
    ax.set_xticks(TICKS)
    ax.set_yticks(TICKS)
    ax.set_aspect('equal', adjustable='box')

    ax.set_title(label, fontsize=TITLE_FS, fontweight='bold', pad=24)
    ax.set_xlabel('Interpreter mean', fontsize=AXIS_FS)
    ax.set_ylabel('Model predicted probability', fontsize=AXIS_FS)
    ax.tick_params(axis='both', labelsize=TICK_FS)

    # R-squared above the top-left corner, outside the plotting area.
    ax.text(0.0, 1.02, f'$R^2$ = {r2:.2f}', transform=ax.transAxes,
            ha='left', va='bottom', fontsize=ANNOT_FS)

    if legend_handles is None:
        legend_handles, legend_labels = ax.get_legend_handles_labels()

# Single shared legend, centered below all four panels (collaborator id=33).
fig.legend(legend_handles, legend_labels, loc='lower center', ncol=4,
           fontsize=LEG_FS, frameon=True, framealpha=0.95, edgecolor='gray')

plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(OUT_DIR / 'hero_scatter_relabeled.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'hero_scatter_relabeled.pdf', bbox_inches='tight')
plt.show()
print('Saved figure to', OUT_DIR)

## 8. Goodness-of-fit test

In [ ]:
import statsmodels.api as sm

# Fit an OLS model for each prediction raster against interpreter mean.
results = {}

for col in model_cols:
    x = df2['interp_mean']
    y = df2[col]

    mask = x.notna() & y.notna()
    x_clean = x[mask]
    y_clean = y[mask]

    X = sm.add_constant(x_clean)
    model = sm.OLS(y_clean, X).fit()

    print("\n" + "=" * 80)
    print(f"OLS summary for: {col}  (n={len(y_clean)})")
    print("=" * 80)
    print(model.summary())

    results[col] = {
        'model': model,
        'rsquared': model.rsquared,
        'adj_rsquared': model.rsquared_adj,
        'f_pvalue': model.f_pvalue,
        'params': model.params,
        'nobs': int(model.nobs)
    }

We conducted ordinary least squares (OLS) regression to test how well interpreter predictions match actual model predictions, and all four models showed statistically significant fits (p < 0.001) with R² values between 0.48-0.62 (model values sampled at the centroid pixel). The regression slopes near 1.0 and intercepts close to 0 indicate good calibration—meaning the interpreters not only correlate well with the models but also predict similar magnitudes of output values.

## 9. Confusion matrices

In [ ]:
# Recode interpreter mean into three categories.
conditions = [
    df2['interp_mean'] <= 0.3,
    df2['interp_mean'] >= 0.7
]
choices = ['0.3 or less', '0.7 or more']

df2['interp_mean_cat'] = np.select(
    conditions,
    choices,
    default='confused_points_0.3to0.7'
)

# Build confusion matrices for each model.
conf_mats = {}

for col in model_cols:
    conditions_model = [
        df2[col] <= 0.3,
        df2[col] >= 0.7
    ]

    df2[f'{col}_cat'] = np.select(
        conditions_model,
        choices,
        default='confused_points_0.3to0.7'
    )

    cm = pd.crosstab(
        df2['interp_mean_cat'],
        df2[f'{col}_cat'],
        rownames=['Interpreter'],
        colnames=[col]
    )

    cm = cm.reindex(
        index=['0.3 or less', 'confused_points_0.3to0.7', '0.7 or more'],
        columns=['0.3 or less', 'confused_points_0.3to0.7', '0.7 or more'],
        fill_value=0
    )

    conf_mats[col] = cm

# Print confusion matrices.
for col, cm in conf_mats.items():
    print(f"\nConfusion matrix for {col}:\n")
    print(cm)

## 10. Accuracy metrics from confusion matrices

In [ ]:
# Compute overall, producer's, and user's accuracy from a confusion matrix.
def accuracy_metrics(cm):
    oa = cm.values.diagonal().sum() / cm.values.sum() * 100
    pa = cm.apply(lambda row: row[row.name] / row.sum() * 100, axis=1)
    ua = cm.apply(lambda col: col[col.name] / col.sum() * 100, axis=0)

    return {
        'Overall Accuracy (%)': oa,
        'Producer Accuracy (%)': pa,
        'User Accuracy (%)': ua
    }


# Print accuracy summaries for each model.
for col, cm in conf_mats.items():
    metrics = accuracy_metrics(cm)

    print(f"\nAccuracy metrics for {col}:")
    print(f"Overall Accuracy: {metrics['Overall Accuracy (%)']:.2f}%")
    print("Producer Accuracy (%):")
    print(metrics['Producer Accuracy (%)'].round(2))
    print("User Accuracy (%):")
    print(metrics['User Accuracy (%)'].round(2))

## 11. Check the number of points used

In [ ]:
# Print valid point counts by model and source group.
print("Point counts by model and source:")
print("=" * 60)

for col in model_cols:
    cert_mask = (df2['source'] == 'cert') & df2['interp_mean'].notna() & df2[col].notna()
    conf_mask = (df2['source'] == 'conf') & df2['interp_mean'].notna() & df2[col].notna()

    n_cert = cert_mask.sum()
    n_conf = conf_mask.sum()
    n_total = n_cert + n_conf

    print(f"\n{col}:")
    print(f"  Base-learners agree (cert):                {n_cert:,}")
    print(f"  Less consensus among base-learners (conf): {n_conf:,}")
    print(f"  Total points:                              {n_total:,}")

print("\n" + "=" * 60)